In [102]:
import math
import time
from pathlib import Path
import torch
import torch.nn as nn
import sentencepiece as spm

#### 加载分词器

In [103]:
model_dir = Path.cwd()/"tokenizers"
zh_sp = spm.SentencePieceProcessor(model_file=str(model_dir/"zh_model.model"))
en_sp = spm.SentencePieceProcessor(model_file=str(model_dir/"en_model.model"))
# 中译英
src_vocab_size = zh_sp.get_piece_size()
tgt_vocab_size = en_sp.get_piece_size()

#### 实现位置编码

In [104]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1024):
        super().__init__()
        if d_model % 2 != 0:
            raise ValueError("当前实现要求 d_model 为偶数")
        self.dropout = nn.Dropout(dropout)
        # 创建位置编码矩阵
        pe = torch.zeros(max_len, d_model)
        # 生成位置索引
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        # 计算不同维度对应的频率
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        # 偶数维使用 sin，奇数维使用 cos
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # 增加批次维度：(1, max_len, d_model)
        pe = pe.unsqueeze(0)
        # 保存为非训练参数的 buffer
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x 的形状：(B, L, D)
        seq_len = x.size(1)
        if seq_len > self.pe.size(1):
            raise ValueError("序列长度超过 max_len")
        # 添加位置信息，再进行 Dropout
        x = x + self.pe[:, :seq_len, :]

        return self.dropout(x)

#### 拼接我们的模型

In [105]:
class TranslationTransformer(nn.Module):
    def __init__(self,src_vocab_size,tgt_vocab_size,d_model=256,nhead=8,num_encoder_layers=3,
        num_decoder_layers=3,dim_feedforward=512,dropout=0.1,max_len=512,src_pad_id=0,tgt_pad_id=0):
        super().__init__()
        self.d_model = d_model
        # 中文和英文的 Token Embedding
        self.src_embedding = nn.Embedding(src_vocab_size,d_model,padding_idx=src_pad_id)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size,d_model,padding_idx=tgt_pad_id)

        # 正余弦位置编码
        self.src_pos_encoding = PositionalEncoding(d_model=d_model,dropout=dropout,max_len=max_len)
        self.tgt_pos_encoding = PositionalEncoding(d_model=d_model,dropout=dropout,max_len=max_len)

        # 调用 PyTorch 官方 Transformer
        self.transformer = nn.Transformer(d_model=d_model,nhead=nhead,num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,dim_feedforward=dim_feedforward,dropout=dropout,batch_first=True)
        # 将特征向量映射到英文词表
        self.output_projection = nn.Linear(d_model,tgt_vocab_size)

    def forward(self,src,decoder_input,src_padding_mask=None,tgt_padding_mask=None,causal_mask=None):
        # src: (B, S)
        # decoder_input: (B, T)
        # Token ID -> Embedding
        src_emb = self.src_embedding(src)
        tgt_emb = self.tgt_embedding(decoder_input)

        # 按原始 Transformer 的缩放方式处理 Embedding
        scale = math.sqrt(self.d_model)
        src_emb = src_emb * scale
        tgt_emb = tgt_emb * scale

        # 添加位置编码
        src_emb = self.src_pos_encoding(src_emb)
        tgt_emb = self.tgt_pos_encoding(tgt_emb)

        # 经过 Encoder 和 Decoder
        output = self.transformer(src=src_emb,tgt=tgt_emb,tgt_mask=causal_mask,src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,memory_key_padding_mask=src_padding_mask)
        # 映射到英文词表的每个 Token
        logits = self.output_projection(output)

        return logits

In [ ]:
# 创建模型
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = TranslationTransformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=256,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=512,
    dropout=0.1,
    max_len=1024,
    src_pad_id=zh_sp.pad_id(),
    tgt_pad_id=en_sp.pad_id()
).to(device)


In [107]:
# 当前 model_dir 是 src/model
project_dir = Path.cwd().parents[1]
print("项目根目录：", project_dir)
data_dir = project_dir / "data" / "sample_for_train_val_test"

class TranslationDataset(Dataset):
    def __init__(self, zh_path, en_path, zh_sp, en_sp, max_len=128):
        with open(zh_path, encoding="utf-8") as f:
            zh_lines = [line.strip() for line in f]

        with open(en_path, encoding="utf-8") as f:
            en_lines = [line.strip() for line in f]

        if len(zh_lines) != len(en_lines):
            raise ValueError("中英文行数不一致！")

        self.samples = []

        for zh, en in zip(zh_lines, en_lines):
            # 将句子转换为 Token ID
            zh_ids = zh_sp.encode(zh, out_type=int)
            en_ids = en_sp.encode(en, out_type=int)

            # 加入句首和句尾标记，并限制序列长度
            zh_ids = [zh_sp.bos_id()] + zh_ids[:max_len - 2] + [zh_sp.eos_id()]
            en_ids = [en_sp.bos_id()] + en_ids[:max_len - 2] + [en_sp.eos_id()]

            self.samples.append((zh_ids, en_ids))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

项目根目录： d:\programs\Transformer-translate


In [108]:
train_dataset = TranslationDataset(
    data_dir / "train_zh.txt",
    data_dir / "train_en.txt",
    zh_sp, en_sp
)

val_dataset = TranslationDataset(
    data_dir / "val_zh.txt",
    data_dir / "val_en.txt",
    zh_sp, en_sp
)

print("训练样本数：", len(train_dataset))
print("验证样本数：", len(val_dataset))

训练样本数： 18000
验证样本数： 1000


In [109]:
def collate_fn(batch):
    src_pad = zh_sp.pad_id()
    tgt_pad = en_sp.pad_id()

    src_sequences, tgt_sequences = zip(*batch)

    def pad_sequences(sequences, pad_id):
        max_len = max(len(seq) for seq in sequences)
        result = torch.full(
            (len(sequences), max_len),
            pad_id,
            dtype=torch.long
        )

        for i, seq in enumerate(sequences):
            result[i, :len(seq)] = torch.tensor(seq)

        return result

    src = pad_sequences(src_sequences, src_pad)
    tgt_full = pad_sequences(tgt_sequences, tgt_pad)

    # 右移一位：输入不包含最后一个 Token，标签不包含第一个 Token
    decoder_input = tgt_full[:, :-1]
    labels = tgt_full[:, 1:].clone()

    # 不计算 Padding 位置的损失
    labels[labels == tgt_pad] = -100

    return {
        "src": src,
        "decoder_input": decoder_input,
        "labels": labels,
        "src_padding_mask": src.eq(src_pad),
        "tgt_padding_mask": decoder_input.eq(tgt_pad),
        "causal_mask": torch.triu(
            torch.ones(
                decoder_input.size(1),
                decoder_input.size(1),
                dtype=torch.bool
            ),
            diagonal=1
        )
    }

In [110]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

print("训练 Batch 数：", len(train_loader))
print("验证 Batch 数：", len(val_loader))

训练 Batch 数： 563
验证 Batch 数： 32


In [111]:
batch = next(iter(train_loader))

for name, tensor in batch.items():
    print(
        f"{name:20s}",
        "shape =", tuple(tensor.shape),
        "dtype =", tensor.dtype
    )

src                  shape = (32, 56) dtype = torch.int64
decoder_input        shape = (32, 59) dtype = torch.int64
labels               shape = (32, 59) dtype = torch.int64
src_padding_mask     shape = (32, 56) dtype = torch.bool
tgt_padding_mask     shape = (32, 59) dtype = torch.bool
causal_mask          shape = (59, 59) dtype = torch.bool


In [112]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

model.train()

# 将一个 Batch 移动到计算设备
src = batch["src"].to(device)
decoder_input = batch["decoder_input"].to(device)
labels = batch["labels"].to(device)

src_padding_mask = batch["src_padding_mask"].to(device)
tgt_padding_mask = batch["tgt_padding_mask"].to(device)
causal_mask = batch["causal_mask"].to(device)

# 1. 前向传播
logits = model(
    src=src,
    decoder_input=decoder_input,
    src_padding_mask=src_padding_mask,
    tgt_padding_mask=tgt_padding_mask,
    causal_mask=causal_mask
)

# 2. 计算预测误差
loss = criterion(
    logits.reshape(-1, logits.size(-1)),
    labels.reshape(-1)
)

# 3. 反向传播并更新参数
optimizer.zero_grad()
loss.backward()

nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

optimizer.step()

print("第一次训练完成，loss =", loss.item())

第一次训练完成，loss = 9.187697410583496


In [113]:
# 使用求和形式，便于准确统计整个数据集的平均 Token 损失
criterion = nn.CrossEntropyLoss(
    ignore_index=-100,
    reduction="sum"
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

# 保存模型的目录
checkpoint_dir = project_dir / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

best_val_loss = float("inf")
num_epochs = 1  # 第一轮先测试完整训练流程

for epoch in range(num_epochs):
    start_time = time.time()

    # ==========================
    # A. 训练阶段
    # ==========================
    model.train()

    train_loss_sum = 0.0
    train_token_count = 0

    for step, batch in enumerate(train_loader, start=1):
        src = batch["src"].to(device)
        decoder_input = batch["decoder_input"].to(device)
        labels = batch["labels"].to(device)

        src_padding_mask = batch["src_padding_mask"].to(device)
        tgt_padding_mask = batch["tgt_padding_mask"].to(device)
        causal_mask = batch["causal_mask"].to(device)

        # 前向传播
        logits = model(
            src=src,
            decoder_input=decoder_input,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask,
            causal_mask=causal_mask
        )

        # 计算当前 Batch 的总损失
        loss_sum = criterion(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1)
        )

        # 统计真正参与损失计算的 Token 数量
        token_count = (labels != -100).sum()

        # 取平均损失用于反向传播
        loss = loss_sum / token_count.clamp_min(1)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += loss_sum.item()
        train_token_count += token_count.item()

        if step % 50 == 0 or step == len(train_loader):
            print(
                f"Epoch {epoch + 1} | "
                f"Step {step}/{len(train_loader)} | "
                f"当前 Batch Loss: {loss.item():.4f}"
            )

    train_loss = train_loss_sum / max(train_token_count, 1)

    # ==========================
    # B. 验证阶段
    # ==========================
    model.eval()

    val_loss_sum = 0.0
    val_token_count = 0

    with torch.no_grad():
        for batch in val_loader:
            src = batch["src"].to(device)
            decoder_input = batch["decoder_input"].to(device)
            labels = batch["labels"].to(device)

            src_padding_mask = batch["src_padding_mask"].to(device)
            tgt_padding_mask = batch["tgt_padding_mask"].to(device)
            causal_mask = batch["causal_mask"].to(device)

            logits = model(
                src=src,
                decoder_input=decoder_input,
                src_padding_mask=src_padding_mask,
                tgt_padding_mask=tgt_padding_mask,
                causal_mask=causal_mask
            )

            loss_sum = criterion(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            )

            token_count = (labels != -100).sum()

            val_loss_sum += loss_sum.item()
            val_token_count += token_count.item()

    val_loss = val_loss_sum / max(val_token_count, 1)

    print(f"\nEpoch {epoch + 1} 结果")
    print(f"训练损失：{train_loss:.4f}")
    print(f"验证损失：{val_loss:.4f}")
    print(f"耗时：{time.time() - start_time:.1f} 秒")

    # ==========================
    # C. 保存最佳模型
    # ==========================
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
                "src_vocab_size": src_vocab_size,
                "tgt_vocab_size": tgt_vocab_size
            },
            checkpoint_dir / "best_transformer.pt"
        )

        print("已保存当前最佳模型。")

Epoch 1 | Step 50/563 | 当前 Batch Loss: 7.4879
Epoch 1 | Step 100/563 | 当前 Batch Loss: 6.8858
Epoch 1 | Step 150/563 | 当前 Batch Loss: 6.5858
Epoch 1 | Step 200/563 | 当前 Batch Loss: 6.6031
Epoch 1 | Step 250/563 | 当前 Batch Loss: 6.3002
Epoch 1 | Step 300/563 | 当前 Batch Loss: 6.3519
Epoch 1 | Step 350/563 | 当前 Batch Loss: 6.4733
Epoch 1 | Step 400/563 | 当前 Batch Loss: 6.4085
Epoch 1 | Step 450/563 | 当前 Batch Loss: 6.2589
Epoch 1 | Step 500/563 | 当前 Batch Loss: 6.3790
Epoch 1 | Step 550/563 | 当前 Batch Loss: 6.1864
Epoch 1 | Step 563/563 | 当前 Batch Loss: 5.9395

Epoch 1 结果
训练损失：6.6563
验证损失：6.2027
耗时：70.4 秒
已保存当前最佳模型。
